In [1]:
import numpy as np
import mayavi.mlab as mlab

# -----------------------------
# Load KITTI calibration file
# -----------------------------
def load_calib(calib_file):
    data = {}
    with open(calib_file, "r") as f:
        for line in f:
            line = line.strip()
            if not line or ":" not in line:
                continue  # skip empty/malformed lines
            key, value = line.split(":", 1)
            data[key] = np.array([float(x) for x in value.strip().split()])

    R0 = data["R0_rect"].reshape(3, 3)
    Tr_velo_to_cam = data["Tr_velo_to_cam"].reshape(3, 4)

    # Homogenize
    R0_4x4 = np.eye(4); R0_4x4[:3,:3] = R0
    Tr_velo_to_cam_4x4 = np.eye(4); Tr_velo_to_cam_4x4[:3,:] = Tr_velo_to_cam

    # Inverse transform: Camera → Velodyne
    cam_to_velo = np.linalg.inv(R0_4x4 @ Tr_velo_to_cam_4x4)
    return cam_to_velo


# -----------------------------
# Parse KITTI label file (camera coords)
# -----------------------------
def load_kitti_labels(label_file):
    cuboids = []
    with open(label_file, "r") as f:
        for line in f:
            parts = line.strip().split(" ")
            obj_type = parts[0]
            if obj_type not in ["Car", "Pedestrian", "Cyclist"]:
                continue  # ignore vans, misc, dontcare

            # Fields: type, truncated, occluded, alpha, bbox2d(4), h, w, l, x, y, z, ry
            h, w, l = map(float, parts[8:11])
            x, y, z = map(float, parts[11:14])
            yaw = float(parts[14])

            cuboids.append({
                "label": obj_type,
                "dimensions": (l, w, h),   # (length, width, height)
                "position": (x, y, z),     # in camera coords
                "yaw": yaw                 # rotation around Y-axis (camera coords)
            })
    return cuboids

# -----------------------------
# Transform cuboid center & yaw to Velodyne coords
# -----------------------------
def transform_cuboid_to_velo(cub, cam_to_velo):
    l, w, h = cub["dimensions"]
    x, y, z = cub["position"]

    center_cam = np.array([x, y, z, 1.0])
    center_velo = cam_to_velo @ center_cam

    # Approximate: keep yaw the same (strictly, need to rotate axis too)
    return {
        "label": cub["label"],
        "dimensions": (l, w, h),
        "position": (center_velo[0], center_velo[1], center_velo[2]),
        "yaw": cub["yaw"]
    }

# -----------------------------
# Draw cuboid in Velodyne coords
# -----------------------------
def draw_cuboid_velo(cuboid, color=(1,1,1), line_width=2):
    l, w, h = cuboid["dimensions"]
    x, y, z = cuboid["position"]
    yaw = cuboid["yaw"]

    # Box corners (bottom center at (x,y,z))
    x_corners = [ l/2,  l/2, -l/2, -l/2,  l/2,  l/2, -l/2, -l/2 ]
    y_corners = [ 0,    0,    0,    0,   -h,   -h,   -h,   -h   ]
    z_corners = [ w/2, -w/2, -w/2,  w/2,  w/2, -w/2, -w/2,  w/2 ]
    corners = np.vstack([x_corners, y_corners, z_corners])

    # Rotate around Y
    R = np.array([[ np.cos(yaw), 0, np.sin(yaw)],
                  [ 0,           1, 0          ],
                  [-np.sin(yaw), 0, np.cos(yaw)]])
    corners = R @ corners
    corners = corners.T + np.array([x, y, z])

    edges = [(0,1),(1,2),(2,3),(3,0),
             (4,5),(5,6),(6,7),(7,4),
             (0,4),(1,5),(2,6),(3,7)]
    for i,j in edges:
        mlab.plot3d([corners[i,0], corners[j,0]],
                    [corners[i,1], corners[j,1]],
                    [corners[i,2], corners[j,2]],
                    color=color, line_width=line_width, tube_radius=None)

# -----------------------------
# Fake prediction by small noise
# -----------------------------
def make_noisy_prediction(cub):
    pred = cub.copy()
    x,y,z = cub["position"]
    l,w,h = cub["dimensions"]

    pred["position"] = (x + np.random.uniform(-0.15,0.15),
                        y + np.random.uniform(-0.05,0.05),
                        z + np.random.uniform(-0.15,0.15))
    pred["dimensions"] = (l*np.random.uniform(0.95,1.05),
                          w*np.random.uniform(0.95,1.05),
                          h*np.random.uniform(0.95,1.05))
    pred["yaw"] = cub["yaw"] + np.random.uniform(-0.05,0.05)
    return pred

# -----------------------------
# Visualization
# -----------------------------
mlab.figure("KITTI GT vs Prediction", bgcolor=(1,1,1), size=(1000,600))

# Paths
lidar_file = "/media/bvb/SSD-1/LUMINET/data/KITTI/object/training/velodyne/000010.bin"
label_file = "/media/bvb/SSD-1/LUMINET/data/KITTI/object/training/label_2/000010.txt"
calib_file = "/media/bvb/SSD-1/LUMINET/data/KITTI/object/training/calib/000010.txt"

# Load LiDAR points
lidar_points = np.fromfile(lidar_file, dtype=np.float32).reshape(-1,4)[:, :3]
mlab.points3d(lidar_points[:,0], lidar_points[:,1], lidar_points[:,2],
              mode="point", color=(0,0,0), scale_factor=1)

# Load calibration + labels
cam_to_velo = load_calib(calib_file)
cuboids_cam = load_kitti_labels(label_file)

# Transform all GT boxes to Velodyne frame
cuboids_velo = [transform_cuboid_to_velo(c, cam_to_velo) for c in cuboids_cam]

# Colors
class_colors = {"Car": (1,0,0), "Pedestrian": (1,1,0), "Cyclist": (0,1,1)}
gt_color = (0,0,1)

# Draw GT + fake prediction
for cub in cuboids_velo:
    draw_cuboid_velo(cub, color=gt_color, line_width=4)   # GT thick blue
    pred = make_noisy_prediction(cub)
    draw_cuboid_velo(pred, color=class_colors[cub["label"]], line_width=2)

mlab.show()
